# 案例 10 · FD 泄漏耗尽句柄，接口 500 (Too many open files)

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

详情页新增模板文件读取逻辑，异常分支未关闭句柄；`LimitNOFILE=256`，约 6 分钟后句柄耗尽，接口返回 500 `TOO_MANY_OPEN_FILES`。

## 练习目标

- 用错误码 `Errno 24` 快速定性
- 用 `lsof -p` 统计句柄类型
- 重启止血 + 提升 LimitNOFILE
- 用 `fd_guard.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-10-fd")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_fd_leak(requests=320)
print("已注入：请求泄漏句柄，当前 fds =", lab.api_stats().get("fds"))


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**句柄使用**

In [ ]:
print("fds =", lab.api_stats().get("fds"), "/ LimitNOFILE =", lab.state["services"]["xhs-note-api"]["nofile"])

**句柄明细**

In [ ]:
print(lab.sh("lsof -p %d | head -20" % lab.state["services"]["xhs-note-api"]["pid"]))

**句柄上限**

In [ ]:
print(lab.sh("ulimit -n"))

**应用日志**

In [ ]:
print(lab.sh("tail -n 5 /var/log/xhs/note-api.log"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
st = lab.api_stats()
svc = lab.state["services"]["xhs-note-api"]
print(f"fds={st.get('fds')} / LimitNOFILE={svc.get('nofile')}")
print("业务请求:", lab.curl(lab.api + "/api/v1/note/8f2a41c9").splitlines()[-1])
print("判断：fds 触顶且以 REG 文件为主 → 文件未关闭导致泄漏")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
svc = lab.state["services"]["xhs-note-api"]
# 1) 提升上限（生产应 65535，沙箱内核生效值 4096）
unit = lab.path("/etc/systemd/system/xhs-note-api.service").read_text()
lab.write("/etc/systemd/system/xhs-note-api.service", unit.replace("LimitNOFILE=256", "LimitNOFILE=65535"))
# 2) 重启释放泄漏句柄
lab.restart_api(port=svc["port"], nofile=4096)
lab.state["faults"].pop("fd", None)
lab._save()
time.sleep(1)
print("重启后 fds:", lab.api_stats().get("fds"))
print("业务请求:", lab.curl(lab.api + "/api/v1/note/8f2a41c9").splitlines()[-1])


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('FD 已回落', bool(lab.api_stats().get('fds', 999) < 100))
lab.check('接口恢复 200', bool('HTTP_CODE=200' in lab.curl(lab.api + '/api/v1/note/8f2a41c9')))

## 步骤 6 · 自动化沉淀

脚本读取 FD 使用率与上限，超 80% 时提升 `LimitNOFILE`、重启释放句柄，最后校验接口 200 与 FD 回落。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_fd_leak(requests=320)
out = lab.auto('fd_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. `lsof` 中 `REG`、`IPv4`、`PIPE` 分别指向哪类泄漏？
2. 为什么提升上限不能替代修代码？
3. 容器环境下如何正确设置 FD 上限？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')